# Vnlp-scale Colab Chat Demo

This notebook records a Llama-compatible Hugging Face checkpoint into a Vnlp-scale store, loads the PyTorch streaming runtime, and launches an interactive Gradio chat UI.

The default model is `TinyLlama/TinyLlama-1.1B-Chat-v1.0`, which is small enough for a typical Colab GPU. Recording is resumable: if Colab disconnects, rerun the recording cell with the same settings.

> Select **Runtime → Change runtime type → GPU** before starting. CPU execution is supported but substantially slower.

In [ ]:
import subprocess
import sys

# "main" is the durable default. During review of a feature PR, replace it with
# that PR branch name to preview unmerged runtime changes.
VNLP_REF = "main"

packages = [
    f"vnlp-scale[torch,tokenizer] @ git+https://github.com/vnlpscale/Vnlp-scale.git@{VNLP_REF}",
    "gradio>=5,<6",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
print("Installed Vnlp-scale and chat-demo dependencies.")

## Configuration

`low` uses a low-rank stage plus a 2-bit residual. It demonstrates stage-aware fused linear execution but takes longer to record than `lossless`.

Set `STORE_DIR` to a Google Drive path to persist the encoded model across Colab sessions.

In [ ]:
from pathlib import Path

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
QUALITY = "low"               # lossless, low, med, or high
STORE_DIR = Path("/content/vnlp-tinyllama-low")
CHUNK_MIB = 16

CACHE_MIB = 2048              # Shared cache for factors, packed codes, scales, and decoded chunks
MAX_STAGES = None             # 1 = low-rank stage only; None = all stored stages
QUANT_BLOCK_ROWS = 256
MAX_CONTEXT_TOKENS = 2048
DEFAULT_MAX_NEW_TOKENS = 64
DEFAULT_TEMPERATURE = 0.7
SYSTEM_PROMPT = "You are a concise and helpful assistant."

print({
    "model": MODEL_ID,
    "quality": QUALITY,
    "store": str(STORE_DIR),
})

## Optional Hugging Face authentication

The default model is public. For a gated model, add an `HF_TOKEN` secret in the Colab **Secrets** panel and grant notebook access.

In [ ]:
import os

try:
    from google.colab import userdata

    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    print("HF_TOKEN is configured.")
else:
    print("No HF_TOKEN configured; public models remain available.")

## Record the model

The operation writes checkpoints at chunk boundaries. A finalized store is reused automatically; an interrupted store resumes when this cell is rerun.

In [ ]:
import json

from vnlp_scale.ingest import record

manifest_path = STORE_DIR / "manifest.json"
already_finalized = False
if manifest_path.is_file():
    try:
        already_finalized = bool(
            json.loads(manifest_path.read_text(encoding="utf-8")).get("finalized")
        )
    except (OSError, json.JSONDecodeError):
        already_finalized = False

if already_finalized:
    print(f"Using finalized store: {STORE_DIR}")
else:
    result = record(
        MODEL_ID,
        str(STORE_DIR),
        quality=QUALITY,
        max_chunk_bytes=CHUNK_MIB * 1024 * 1024,
        checkpoint_every=4,
        overwrite=False,
        progress=print,
    )
    print(json.dumps(result["summary"], indent=2))

In [ ]:
from vnlp_scale.store import StoreReader

with StoreReader(STORE_DIR) as reader:
    verification = reader.verify(checksums=False)
    summary = reader.summary()

print("Verification:", verification)
print("Bits/parameter:", round(summary["bits_per_parameter"], 3))
print("Compression vs FP16:", round(summary["compression_vs_fp16"], 3))

## Load the streaming runtime

On CUDA, Vnlp-scale automatically chooses BF16 when supported, otherwise FP16. Attention falls back from FlashAttention-2 to PyTorch SDPA when `flash-attn` is not installed.

The stage-aware path uses low-rank factors directly and, when Triton is available, consumes packed 2/4/8-bit residual codes without materializing a complete weight matrix.

In [ ]:
import inspect

import torch
from transformers import AutoTokenizer
from vnlp_scale.engine_torch import TorchLlamaEngine

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = "auto" if device == "cuda" else "float32"
torch.set_float32_matmul_precision("high")

tokenizer = AutoTokenizer.from_pretrained(STORE_DIR, local_files_only=True)
if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
    tokenizer.pad_token = tokenizer.eos_token

engine_options = {
    "device": device,
    "dtype": dtype,
    "cache_bytes": CACHE_MIB * 1024 * 1024,
    "max_stages": MAX_STAGES,
    "verify": False,
}

# Keep the notebook compatible with older main-branch revisions while enabling
# the optimized controls automatically after the stage-aware runtime is merged.
parameters = inspect.signature(TorchLlamaEngine.from_store).parameters
optional_options = {
    "attention_backend": "auto",
    "compile_resident": True,
    "cuda_graph_decode": True,
    "fused_linear": True,
    "quant_block_rows": QUANT_BLOCK_ROWS,
    "triton_quant": True,
}
engine_options.update({
    name: value for name, value in optional_options.items() if name in parameters
})

engine = TorchLlamaEngine.from_store(STORE_DIR, **engine_options)
print("Device:", device)
print("Optimizations:", engine.optimization_report())

## Warm up

The first call may compile resident operations and a Triton packed-quant kernel. This warm-up keeps the first chat turn from paying all initialization overhead.

In [ ]:
def encode_messages(messages):
    try:
        token_ids = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
        )
    except Exception:
        fallback = "\n".join(
            f"{item['role'].capitalize()}: {item['content']}" for item in messages
        ) + "\nAssistant:"
        token_ids = tokenizer.encode(fallback, add_special_tokens=True)
    return list(token_ids)[-MAX_CONTEXT_TOKENS:]


warmup_ids = encode_messages([
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "Say hello in one sentence."},
])
warmup = engine.generate(warmup_ids, 2, greedy=True)
print("Warm-up tokens/s:", round(warmup["tokens_per_second"], 3))
print("Provider stats:", warmup["provider_stats"])

## Launch the chat demo

Each turn rebuilds the prompt from visible conversation history. This is intentionally simple and does not yet retain a cross-turn KV cache.

In [ ]:
import threading

import gradio as gr

generation_lock = threading.Lock()


def normalized_history(history):
    messages = []
    for item in history or []:
        if isinstance(item, dict):
            role = item.get("role")
            content = item.get("content")
        else:
            role = getattr(item, "role", None)
            content = getattr(item, "content", None)
        if role in {"user", "assistant"} and isinstance(content, str):
            messages.append({"role": role, "content": content})
    return messages


def trim_at_eos(token_ids):
    eos_id = tokenizer.eos_token_id
    if eos_id is not None and eos_id in token_ids:
        return token_ids[: token_ids.index(eos_id) + 1]
    return token_ids


def respond(message, history, max_new_tokens, temperature, sample):
    message = (message or "").strip()
    if not message:
        return "Enter a message."

    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    messages.extend(normalized_history(history))
    messages.append({"role": "user", "content": message})
    prompt_ids = encode_messages(messages)

    with generation_lock:
        result = engine.generate(
            prompt_ids,
            int(max_new_tokens),
            greedy=not bool(sample),
            temperature=max(float(temperature), 1e-5),
        )

    generated_ids = trim_at_eos(list(result["tokens"]))
    answer = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    if not answer:
        answer = "(The model returned only special tokens.)"

    optimizations = result.get("optimizations", {})
    provider = result.get("provider_stats", {})
    footer = (
        f"\n\n---\n"
        f"`{result['tokens_per_second']:.2f} tok/s` · "
        f"`{len(generated_ids)} tokens` · "
        f"`{optimizations.get('dtype', 'unknown')}` · "
        f"`attention={optimizations.get('attention', 'unknown')}` · "
        f"`fused={optimizations.get('stage_aware_linear', False)}` · "
        f"`triton_stages={provider.get('triton_quant_stages', 0)}`"
    )
    return answer + footer


demo = gr.ChatInterface(
    fn=respond,
    chatbot=gr.Chatbot(type="messages", height=560),
    textbox=gr.Textbox(
        placeholder="Ask TinyLlama something…",
        container=False,
        scale=7,
    ),
    title="Vnlp-scale Chat Demo",
    description=(
        "Bounded-memory chat inference from a checksummed, chunk-addressable model store."
    ),
    additional_inputs=[
        gr.Slider(
            1,
            256,
            value=DEFAULT_MAX_NEW_TOKENS,
            step=1,
            label="Maximum new tokens",
        ),
        gr.Slider(
            0.05,
            1.5,
            value=DEFAULT_TEMPERATURE,
            step=0.05,
            label="Temperature",
        ),
        gr.Checkbox(value=True, label="Sample"),
    ],
    examples=[
        ["Explain why compressed-stage matrix multiplication saves memory."],
        ["Write a short Python function that computes cosine similarity."],
        ["日本語で自己紹介してください。"],
    ],
)

demo.queue(default_concurrency_limit=1).launch(share=True, debug=False)

## Cleanup

Run this cell before changing model or store settings.

In [ ]:
try:
    demo.close()
except Exception:
    pass

engine.close()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Closed the demo and released runtime caches.")

## Troubleshooting

- **Out of memory:** reduce `CACHE_MIB`, `CHUNK_MIB`, `QUANT_BLOCK_ROWS`, or `MAX_CONTEXT_TOKENS`.
- **Recording interrupted:** rerun the recording cell. Committed chunks are skipped.
- **Triton compilation fails:** the runtime disables the packed kernel and uses the bounded row-block fallback.
- **Unsupported model:** use a standard Llama or Mistral checkpoint without custom RoPE scaling or attention/MLP biases.
- **Slow first response:** rerun the warm-up cell after loading the engine.